# PyBADS Example 6: Periodic variables

In this example, we will show how to run PyBADS on a target with periodic variables, such as angles.

This notebook is Part 6 of a series of notebooks in which we present various example usages for BADS with the PyBADS package.
The code used in this example is available as a script [here](./scripts/pybads_example_6_periodic_variables.py).

In [1]:
import numpy as np
from pybads import BADS

## 0. Periodic variables

Some model parameters are *periodic*: an angle, a phase or a time of day describes the same state again after a whole period $p$ (e.g., $p = 2\pi$ for an angle in radians). The target function is then periodic along such a variable $x_d$:

$$
f(x_1, \ldots, x_d + p, \ldots, x_D) = f(x_1, \ldots, x_d, \ldots, x_D).
$$

We tell PyBADS which variables are periodic with the option `periodic_vars`, a list of their indices (counted from 0, as any Python index). The hard bounds `lower_bounds` and `upper_bounds` of a periodic variable need to be finite, and are its period: $p = \text{upper\_bounds}_d - \text{lower\_bounds}_d$. PyBADS takes the two bounds as the same point and *wraps* the variable around them, so that a step past one bound comes back in from the other. PyBADS's model of the target, a Gaussian process, is periodic along the variable as well.

## 1. Problem setup

We optimize a function of $D = 4$ variables, the sum of [Rosenbrock's banana function](https://en.wikipedia.org/wiki/Rosenbrock_function) of the first two, as in [Example 1](./pybads_example_1_basic_usage.ipynb), and of a cosine of each of the last two:

$$
f(\mathbf{x}) = \text{Rosenbrock}(x_1, x_2) + \cos\left(\frac{\pi x_3}{2}\right) + \cos\left(\pi x_4\right) + 2.
$$

The function is periodic along $x_3$, with period 4, and along $x_4$, with period 2. Its global minimum is $f^\star = 0$, at $x_1 = x_2 = 1$, $x_3 = \pm 2$ and $x_4 = \pm 1$, where each cosine equals $-1$.

We set the hard bounds of $x_3$ and $x_4$ to one period each, $[-2, 2]$ and $[-1, 1]$, and name them as periodic with `periodic_vars = [2, 3]`: they are the third and fourth variables, of indices 2 and 3 in Python. The bounds of a periodic variable only mark where its period is cut, not a range of extreme values, so we set its plausible bounds to its hard bounds as well. For $x_1$ and $x_2$, which are not periodic, we specify wide hard bounds and tighter plausible bounds, as in Example 1.

We start the optimization from $\mathbf{x}_0 = (-3, -3, -1, -1)$, and fix the random seed of the run with `random_seed`, so that it is reproducible.

In [2]:
def rosenbrocks_fcn(x):
    """Rosenbrock's 'banana' function in any dimension."""
    x_2d = np.atleast_2d(x)
    return np.sum(
        100 * (x_2d[:, 0:-1] ** 2 - x_2d[:, 1:]) ** 2
        + (x_2d[:, 0:-1] - 1) ** 2,
        axis=1,
    )


def periodic_fcn(x):
    """Rosenbrock's function of x_1, x_2, plus a cosine of x_3, of period 4,
    and one of x_4, of period 2."""
    x_2d = np.atleast_2d(x)
    return (
        rosenbrocks_fcn(x_2d[:, 0:2])
        + np.cos(np.pi * x_2d[:, 2] / 2)
        + np.cos(np.pi * x_2d[:, 3])
        + 2
    )


lower_bounds = np.array([-10, -5, -2, -1])
upper_bounds = np.array([5, 10, 2, 1])
plausible_lower_bounds = np.array([-2, -2, -2, -1])
plausible_upper_bounds = np.array([2, 2, 2, 1])
x0 = np.array([-3, -3, -1, -1])  # Starting point

options = {
    "periodic_vars": [2, 3],  # The third and fourth variables are periodic
    "random_seed": 0,  # Makes the run reproducible
}

## 2. Run the optimization

We run `bads` with the user-defined `options`. PyBADS lists the periodic variables when the `BADS` object is created.

In [3]:
bads = BADS(
    periodic_fcn,
    x0,
    lower_bounds,
    upper_bounds,
    plausible_lower_bounds,
    plausible_upper_bounds,
    options=options,
)
optimize_result = bads.optimize()

Variables (index) defined with periodic boundaries: [2, 3]
Beginning optimization of a DETERMINISTIC objective function

 Iteration    f-count         f(x)           MeshScale          Method             Actions
     0           2           14417               1                                 Uncertainty test
     0          10         73.9745               1         Initial mesh            Initial points
     1          16         72.1486               1       Successful poll           Train
     2          17         5.63746               1     Successful search (ES-wcm)        
     2          19         4.41123               1     Successful search (ES-wcm)        
     2          25          4.1116               1     Incremental search (ES-wcm)        
     2          32          4.1116             0.5         Refine grid             Train
     3          35         3.67243             0.5     Successful search (ES-wcm)        
     3          37         3.30242             0.5 

## 3. Results and conclusions

In [4]:
x_min = optimize_result["x"]
fval = optimize_result["fval"]

print(f"BADS minimum at: x_min = {x_min.flatten()}, fval = {fval:.4g}")
print(
    f"total f-count: {optimize_result['func_count']}, time: {round(optimize_result['total_time'], 2)} s"
)

BADS minimum at: x_min = [ 0.9970266   0.99405845  1.99980797 -0.99994354], fval = 8.904e-06
total f-count: 185, time: 4.12 s


The true global minimum is at $\mathbf{x}^\star = (1, 1, \pm 2, \pm 1)$, where $f^\star = 0$.

PyBADS reports a periodic variable within its period, between `lower_bounds` and `upper_bounds`. Since the two bounds of a periodic variable are the same point, a minimum on them can come out near either end of the period: $x_3$ at or just above $-2$, or just below $2$, and $x_4$ at or just above $-1$, or just below $1$.

### Remarks

- Here the minimum along $x_3$ and $x_4$ lies on the bounds, where the period is cut. For a periodic variable, a bound is not an edge of the domain: the minimum lies at the bottom of a valley of the target that extends to both sides of the bound, and PyBADS reaches it from either side, moving across the bound. A run can likewise take the short way to a minimum near one bound from a point near the other, across the bounds rather than through the whole period.

- PyBADS takes the width of the hard bounds of a periodic variable as its period, so set them one period apart, such as `0` and `2 * np.pi`, or `-np.pi` and `np.pi`, for an angle in radians. With bounds that are not a period apart, the target would jump where PyBADS wraps the variable around.

- For more information on periodic variables, see the [PyBADS FAQ](https://acerbilab.github.io/pybads/faq.html#faq-does-pybads-support-periodic-variables-such-as-angles).